# Dask and virtual environment patterns

This notebook demonstrates ways to make and manage a custom virtual environment (venv) in EASI and with Dask clusters.
A key tenant is to efficiently append to or alter the python geospatial environments available in the EASI pre-built images.
The EASI pre-built images include a managed set of package versions (C libraries and python packages).
Adding your own or additional packages without consideration for the image environment can lead to version conflicts and software errors.

The patterns shown here should be sufficient for users to adapt to their workflows.

| Cluster | Method | Package distribution | When to use |
|--|--|--|--|
| Local | [Build a venv](#Build-a-venv-locally) | EASI-aware `uv` script | Add packages and manage dependencies |
| Local | [sys.path injection](#Local-cluster---sys.path-injection) | Use the local venv | Local cluster use cases and smaller scale development/testing |
| Gateway | [Upload file](#Gateway-cluster---Upload-file) | Upload files to each worker | Custom requirements, scripts or eggs/zips |
| Gateway | [PipInstall](#Gateway-cluster---Pip-Install) | Pip on each worker | Pip install a few packages |
| Gateway | [Venv setup script](#Gateway-cluster---Venv-setup-script) | UploadFile + `uv` script | Full control, constraints/overrides |
| Gateway | [Install from eggs or wheels](#Install-from-eggs-or-wheels) | Eggs or Wheels | Efficient for known package versions |
| Any | A custom image build | Pre-built environment | Mature large-scale or repeated workflows |

For each dask demonstration we:
- start a new cluster
- apply the plugin(s)
- scale the cluster
- confirm that the plugin(s) have been applied to workers
- close the cluster

The time taken taken to apply plugins and scale the cluster is captured for each demonstration for information and comparison.
However, this time includes worker start up time, which can be highly variable and likely much greater than the time to apply plugins in these examples.

## Appendix
- [Capturing logs from dask workers](#Capturing-logs-from-dask-workers)

## Setup

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import time
import logging
from dask.distributed import PipInstall, WorkerPlugin, UploadFile
from distributed.diagnostics.plugin import ForwardOutput
from dask_gateway import GatewayCluster


# --- User  ---
VENV_NAME = "myenv"
PACKAGES = "tifftools"   # space-separated; something NOT in the base image
VENV_BASE = str(Path.home() / "venvs")
VENV_PATH = Path(f"{VENV_BASE}/{VENV_NAME}")

easi_venv_local = Path("/opt/easi-venv-setup.sh")  # Newer EASI images
easi_venv_github = Path.home() / "easi-cookbook/venv-setup/easi-venv-setup.sh"  # Or clone this repo to your home dir
SCRIPT_PATH = str(easi_venv_local) if easi_venv_local.exists() else str(easi_venv_github)

print(f"VENV_PATH  : {VENV_PATH}")
print(f"SCRIPT_PATH: {SCRIPT_PATH}")
print(f"Venv exists: {VENV_PATH.exists()}")

## Build a venv locally

Run the `easi-venv-setup.sh` script to create or add to a local virtual environment. Adjust parameters as desired.

See `easi-venv-setup.sh --help`, easi-cookbook venv-setup notebooks (including this notebook) and the EASI user guide for documentation and more details on parameters including how to use constraints and overrides.

For this notebook, we don't need to activate the kernel in our jupyter session becuase we're not actively using the extra packages installed.

In [ ]:
# Parameters are passed to the shell script as environment variables
result = subprocess.run(
    ["bash", SCRIPT_PATH],
    env={**os.environ, "PACKAGES": PACKAGES, "VENV_NAME": VENV_NAME, "INSTALL_KERNEL": "false"},
    capture_output=False,
    check=True,
)

## Worker Plugins

Dask WorkerPlugin classes are used with `cluster.register_plugin(plugin)`.
Registered plugins will be applied to new and existing workers.

These custom plugins are used in the notebook and can be adapted for your needs.

In [ ]:
class SysPathPlugin(WorkerPlugin):
    """Insert a venv's site-packages to sys.path on worker startup.
    Suitable for Local and Gateway clusters.

    Parameters
        venv_path     base venv path on a worker (str)
    """
    name = "SysPathPlugin"
    idempotent = True

    def __init__(self, venv_path):
        self.venv_path = str(venv_path)  # str|Path
        self.logger = logging.getLogger('distributed.worker')

    async def setup(self, worker):
        sp = next(Path(self.venv_path).glob("lib/python*/site-packages"), None)
        if sp and sp.exists():
            self.site_packages = str(sp)
            self.logger.info(f"[{self.name}] Worker site-packages: {sp}")
        else:
            self.site_packages = self.venv_path
            self.logger.warning(f"[{self.name}] No site-packages found under {self.venv_path}. Will use {self.venv_path} directly")
        if self.site_packages not in sys.path:
            sys.path.insert(0, self.site_packages)


class VenvSetupPlugin(WorkerPlugin):
    """Install packages via easi-venv-setup.sh on worker startup.

    For convenience, each path (script, constraints etc) is checked if it exists.
    If not, then check if it exists in the worker's local directory (from
    UploadFile).

    Parameters
        see easi-venv-setup.sh script
    """
    name = "VenvSetupPlugin"
    idempotent = True

    def __init__(self,
                 packages,
                 venv_base="/tmp",
                 venv_name="venv",
                 constraints="/conf/constraints.txt",
                 overrides=None,
                 torch_backend="cpu",
                ):
        self.packages = packages  # space-separated list of packages
        self.venv_base = str(venv_base)  # str|Path
        self.venv_name = venv_name  # str
        self.constraints = constraints  # str (space-separated) | List of file paths
        self.overrides = overrides  # str (space-separated) | List of file paths
        self.torch_backend = torch_backend  # str
        self.script_path = "/opt/easi-venv-setup.sh"
        if isinstance(constraints, (str, Path)):
            self.constraints = str(constraints).split()
        if isinstance(overrides, (str, Path)):
            self.overrides = str(overrides).split()
        self.logger = logging.getLogger('distributed.plugin')

    def _resolve_paths(self, worker, paths) -> list:
        """Check if each path exists.
        If not, check if the file is in the worker.local_directory and use that.
        Return list (possibly empty) contains valid paths.

        Parameters
            paths    str|Path|List[str,Path]
        """
        out = []
        if isinstance(paths, (str, Path)):
            paths = [paths]
        for p in paths:
            p = Path(p)
            if p.exists():
                out.append(str(p))
            else:
                up = Path(worker.local_directory) / p.name
                if up.exists():
                    out.append(str(up))
                else:
                    self.logger.warning(f"[{self.name}] File not found at {p} nor {up}")
        return out

    async def setup(self, worker):
        """Uses script and ancillary files from image or upload directory, creates venv, injects path.""" 
        script_path = self._resolve_paths(worker, self.script_path)[0]
        if not script_path:
            raise RuntimeError(f"[{self.name}] Script \"{Path(self.script_path).name}\" not found on worker. Please upload it.")
        constraints = self._resolve_paths(worker, self.constraints)
        overrides = self._resolve_paths(worker, self.overrides)

        # Run script to create venv
        env = {
            **os.environ,
            "PACKAGES": self.packages,
            "VENV_NAME": self.venv_name,
            "VENV_BASE": self.venv_base,
            "TORCH_BACKEND": self.torch_backend,
            "INSTALL_KERNEL": "false",
        }
        if constraints:
            env["CONSTRAINTS"] = " ".join(constraints)
        if overrides:
            env["OVERRIDES"] = " ".join(overrides)

        script_vars = ["PACKAGES", "VENV_NAME", "VENV_BASE", "TORCH_BACKEND", "INSTALL_KERNEL", "CONSTRAINTS", "OVERRIDES"]
        cmd = [f"{k}=\'{env[k]}\'" for k in script_vars if k in env] + ["bash", str(script_path)]
        self.logger.info(f"[{self.name}] Run script: {' '.join(cmd)}")

        # Ensure only one uv instance per worker at a time
        # async with Lock(socket.gethostname()):
        result = subprocess.run(
            ["bash", str(script_path)],
            env=env,
            capture_output=True,
            text=True,
        )
        self.logger.info(f"{result.stdout}")
        if result.returncode != 0:
            raise RuntimeError(f"[{self.name}] Script failed (exit {result.returncode}): {result.stderr}")

        # Inject venv site-packages into sys.path
        sp = next(Path(self.venv_base).glob(f"{self.venv_name}/lib/python*/site-packages"), None)
        if sp and sp.exists():
            self.site_packages = str(sp)
            self.logger.info(f"[{self.name}] Worker site-packages: {sp}")
        if self.site_packages not in sys.path:
            sys.path.insert(0, self.site_packages)


class PipInstallWheels(WorkerPlugin):
    """Install all wheels found in the worker's local_directory

    Parameters
        pip_options    list of options accepted by python pip (List[str])
    """

    name = "PipInstallWheels"
    idempotent = True

    def __init__(self, pip_options):
        self.pip_options = pip_options
        self.logger = logging.getLogger('distributed.plugin')

    async def setup(self, worker):
        import socket
        from distributed.lock import Lock

        wheels = [str(x) for x in Path(worker.local_directory).glob("*.whl")]
        if len(wheels) == 0:
            raise RuntimeError(f"[{self.name}] No wheels found in {worker.local_directory}. Nothing to install")

        # Ensure only one pip instance per worker at a time
        # async with Lock(socket.gethostname()):
        cmd = ["pip3", "install"] + self.pip_options + wheels
        self.logger.info(f"[{self.name}] {' '.join(cmd)}")
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
        )
        self.logger.info(f"{result.stdout}")
        if result.returncode != 0:
            raise RuntimeError(f"[{self.name}] Pip failed (exit {result.returncode}): {result.stderr}")


### Helper functions

These are convenience functions for interrogating workers and are used to confirm that our plugins have worked. Please adapt them as necessary.

In [ ]:
def test_import(pkgs: str) -> dict:
    """Try to import the given package(s) in the default python environment.
    This tests whether packages are installed and sys.path is defined correctly.

    Parameters
        pkgs    str | List[str]
    Return
        result  Dict[pkg: {version:str|None, file:str|None, error:exception|None}]   
    """
    import importlib
    if isinstance(pkgs, str):
        pkgs = [pkgs]
    result = {}
    for pk in pkgs:
        try:
            mod = importlib.import_module(pk)
            result[pk] = {
                "version": getattr(mod, "__version__", "?"),
                "file": getattr(mod, "__file__", "?"),
                "error": None
            }
        except ImportError as e:
            result[pk] = {
                "version": None,
                "file": None,
                "error": e
            }
    return result


def test_path(paths) -> dict:
    """Test whether each path exists on the workers.

    Parameters:
        paths    str | Path | List[str,Path]
    Return:
        result   Dict[path: bool]
    """
    from pathlib import Path
    if isinstance(paths, (str, Path)):
        paths = [paths]
    result = {}
    for pth in paths:
        result[str(pth)] = Path(pth).exists()
    return result


def test_localdirectory() -> dict:
    """Return a list of the contents of a worker's local directory

    Return
        result   List[str]
    """
    from pathlib import Path
    from dask.distributed import get_worker
    localdir = Path(get_worker().local_directory)
    return [str(f) for f in sorted(localdir.iterdir())]


def test_syspath() -> list:
    """Return a worker's python sys.path

    Return
        result   List[str]
    """
    import sys
    return sys.path

---
## Local cluster - sys.path injection

**How it works:**
- Register a worker plugin that inserts the venv path to the worker's python sys.path
- Run a test import on a worker to confirm

**When to use:**
- Dask workers use the local file system and venv directly
- Manage the local venv separately and efficiently with the `easi-venv-setup.sh` script

In [ ]:
from dask.distributed import LocalCluster


# Create
cluster_a = LocalCluster(n_workers=1)
client_a = cluster_a.get_client()


# Register plugins and scale cluster

start = time.time()

# client_a.register_plugin(ForwardOutput())  # optional, for debugging
client_a.register_plugin(SysPathPlugin(VENV_PATH))

cluster_a.scale(4)
cluster_a.wait_for_workers(4)

elapsed = time.time() - start
print(f"[Local cluster] Scale and apply plugin time: {elapsed:.1f}s for 4 workers")


# Confirm
# Use the first item of PACKAGES as the test import (replace - with _)
test_pkg = PACKAGES.split()[0].replace("-", "_")
future = client_a.submit(test_import, test_pkg)
print()
print("[Local cluster] Worker import test:")
print(future.result())

future = client_a.submit(test_syspath)
print()
print("[Local cluster] Worker sys.path test:")
for p in future.result():
    print(p)

client_a.close()
cluster_a.close()

---
## Gateway cluster - Upload file

**How it works:**
- `UploadFile` plugin copies a given file to each worker's "dask working directory".
- A worker's working directory is named for the worker (so is different for each worker).
- An optional `load=True (default)` can automatically import `*.py`, `*.zip` or `*.egg` files.
- Each `UploadFile()` plugin requires a different registered name otherwise they overwrite each other

**When to use:**
- Upload custom files for use by workers

In [ ]:
# Create cluster, with one worker to warm it up

print("Start cluster")
start = time.time()

cluster_b = GatewayCluster(worker_cores=2, worker_memory=4)
client_b = cluster_b.get_client()
cluster_b.scale(1)
client_b.wait_for_workers(1)

elapsed = time.time() - start
print(f"[Gateway cluster] Startup time: {elapsed:.1f}s for 1 workers\n")


# Setup

# Test with the local system constraint files
system_constraints = ["/conf/constraints.txt", "/conf/no-binary.txt"]


# Register plugins and scale cluster
# - we use `load=False` because the example files are not python loadable

print("Register plugins and scale cluster")
start = time.time()

# client_b.register_plugin(ForwardOutput())  # optional, for debugging
for path in system_constraints:
    if Path(path).exists():
        print(f"-> Upload file: {path}") 
        name = f"upload_file-{Path(path).name.lower()}"
        client_b.register_plugin(UploadFile(path, load=False), name)

cluster_b.scale(4)
client_b.wait_for_workers(4)

elapsed = time.time() - start
print(f"[Gateway cluster] Scale and upload file time: {elapsed:.1f}s for 4 workers")


# Confirm

future = client_b.submit(test_localdirectory)
print()
print("[Gateway cluster] Worker local directory contents:")
for p in future.result():
    print(p)

client_b.close()
cluster_b.close()

---
## Gateway cluster - Pip Install

**How it works:** 
- Use dask's `PipInstall` plugin with `--target` to install to a writeable directory on a worker
- Inject target path into `sys.path` on each worker

**When to use:**
- Install a few extra packages on workers
- Can version match with local venv or apply tailored packages/versions for a specific worker docker image
- Each worker downloads from PyPI independently, which could create a network bottleneck at scale

**Caveats:**
- Worker system python packages/version may not match local system+venv packages depending on the worker docker image
- However, the default EASI images and (and jupyter + worker pairs) do have consistent common package versions so defaults should work well for most users

**Why `--target` is needed on EASI:**
- EASI gateway workers run in **locked virtualenvs** - standard pip installs fail
- `--user` flag also fails - user site-packages are not writable by the PipInstall process
- `--target=/tmp/site-packages` installs to worker's tmp directory (writable)

In [ ]:
# Create cluster, with one worker to warm it up

print("Start cluster")
start = time.time()
cluster_c = GatewayCluster(worker_cores=2, worker_memory=4)
client_c = cluster_c.get_client()
cluster_c.scale(1)
client_c.wait_for_workers(1)
elapsed = time.time() - start
print(f"[Gateway cluster] Startup time: {elapsed:.1f}s for 1 workers")


# Setup

# Optional: capture versions from local venv
venv_python = f"{VENV_PATH}/bin/python"
frozen_output = subprocess.check_output(
    ["uv", "pip", "freeze", "--python", venv_python, "--color=never"],
    text=True,
)
frozen_packages = [line.strip() for line in frozen_output.splitlines()]
print()
print(f"Captured {len(frozen_packages)} package versions")
print(frozen_packages)

# Target directory on worker
# Writable dirs include /tmp, /wk, /home/odc
TARGET_DIR = "/tmp/site-packages"

# Pip options here follow those of easi-venv-setup.py
# Additional constraints/overrides files can be uploaded with cluster.register_plugin(UploadFile(...))
pip_options = [
    "--no-build-isolation",
    "--no-deps",
    f"--target={TARGET_DIR}",
]

# Test whether system constraints files are available on workers
# If not we ignore them in this example but a full implementation may upload these files
# Or upload your custom constraints files
system_constraints = ["/conf/constraints.txt", "/conf/no-binary.txt"]
future = client_c.submit(test_path, system_constraints)

print()
print("[Gateway cluster] Worker path test:")
for path, exists in future.result().items():
    print(f"  {path} exists: {exists}")
    if exists:
        pip_options.append(f"--constraint={path}")


# Register plugin and scale cluster

print()
print("Register plugins and scale cluster")
start = time.time()

# client_c.register_plugin(ForwardOutput())  # optional, for debugging
client_c.register_plugin(PipInstall(packages=frozen_packages, pip_options=pip_options))
client_c.register_plugin(SysPathPlugin(TARGET_DIR))
cluster_c.scale(4)
client_c.wait_for_workers(4)

elapsed = time.time() - start
print(f"[Gateway cluster] Scale and PipInstall time: {elapsed:.1f}s for 4 workers")


# Confirm

test_pkg = PACKAGES.split()[0].replace("-", "_")
future = client_c.submit(test_import, test_pkg)
print()
print("[Gateway cluster] Worker import test:")
print(future.result())

future = client_c.submit(test_syspath)
print()
print("[Local cluster] Worker sys.path test:")
for p in future.result():
    print(p)

client_c.close()
cluster_c.close()

---
## Gateway cluster - Venv setup script

**How it works:**
- Upload any custom constraints or overrides files
- Upload and run the `easi-venv-setup.sh` script

**When to use:**
- Efficiently create a venv on workers using `uv` dependency resolution
- Should be faster and more customisable than `PipInstall`

In [ ]:
# Create cluster, with one worker to warm it up

print("Starting cluster...")
start = time.time()
cluster_d = GatewayCluster(worker_cores=2, worker_memory=4)
client_d = cluster_d.get_client()
cluster_d.scale(1)
client_d.wait_for_workers(1)
elapsed = time.time() - start
print(f"[Gateway cluster] Startup time: {elapsed:.1f}s for 1 workers")


# Setup
# Check that the script, the system constraint files and any custom constraint/override files are on the worker image
# If not then upload them (str or Path)
# The script is required so we make sure we upload a copy if required
#
# System and custom constraint files
#   The /conf/constraints.txt file was used to resolve the system package dependencies - its best to include this with any custom constraint files
#   The /conf/no-binary.txt file is hardwired in the script as its a core system constraint - we check that it exists but don't need to pass it in
#   Use a custom constraint file to resolve your additional package dependencies
#   Use a custom overrides file to override any known dependency conflicts (see uv pip install documentation)

system_constraints = ["/conf/constraints.txt", "/conf/no-binary.txt"]
my_constraints = []
my_overrides = []

future = client_d.submit(test_path, [SCRIPT_PATH] + system_constraints + my_constraints + my_overrides)
print()
print("[Gateway cluster] Worker path test:")
for path, exists in future.result().items():
    print(f"  {path} exists: {exists}")
    # If the file is not on the worker but is local then upload it
    if not exists and Path(path).exists():
        print(f"  -> Upload file: {path}") 
        name = f"upload_file-{Path(path).name.lower()}"
        client_d.register_plugin(UploadFile(path, load=False), name)


# Register plugins and scale cluster

print()
print("Register plugins and scale cluster")
start = time.time()

# client_d.register_plugin(ForwardOutput())  # optional, for debugging
client_d.register_plugin(VenvSetupPlugin(
    PACKAGES,
    constraints=["/conf/constraints.txt"] + my_constraints,
    overrides=my_overrides,
))

cluster_d.scale(4)
client_d.wait_for_workers(4)

elapsed = time.time() - start
print(f"[Gateway cluster] Scale and venv setup time: {elapsed:.1f}s for 4 workers")


# Confirm

future = client_d.submit(test_localdirectory)
print()
print("[Gateway cluster] Worker local directory contents:")
for p in future.result():
    print(p)

test_pkg = PACKAGES.split()[0].replace("-", "_")
future = client_d.submit(test_import, test_pkg)
print()
print("[Gateway cluster] Worker import test:")
print(future.result())

future = client_d.submit(test_syspath)
print()
print("[Local cluster] Worker sys.path test:")
for p in future.result():
    print(p)

client_d.close()
cluster_d.close()

---
## Gateway cluster - Install from eggs or wheels

The `PipInstall` and `VenvSetupPlugin` methods require workers to each download and install packages from the internet. If scaling to many 100-1000s of workers then this may bottleneck.

An alternative for large scaling workflows is to manage your own set of runnable or installable packages. Workers could get these through `UploadFile` or by reading from a shared storage location, such as S3.

The `UploadFile` method has a `load` option that can read and import (`importlib.import_module`) any `*.py`, `*.egg`, `*.zip` or `*.pyz` files ([ref: import_file](https://github.com/dask/distributed/blob/main/distributed/utils.py)). *We don't show an example of this here as we don't have any eggs available and it should be reasonably straightforward*.

Python **wheels** are not yet directly supported by dask plugins. However, we can create our own `WorkerPlugin` that `pip install`s from wheels. Here is an example inspired by this [issue](https://github.com/dask/distributed/issues/6202).

In [ ]:
# Create cluster, with one worker to warm it up

print("Start cluster")
start = time.time()
cluster_e = GatewayCluster(worker_cores=2, worker_memory=4)
client_e = cluster_e.get_client()
cluster_e.scale(1)
client_e.wait_for_workers(1)
elapsed = time.time() - start
print(f"[Gateway cluster] Startup time: {elapsed:.1f}s for 1 workers")


# Setup
# Optional: capture versions from local venv
venv_python = f"{VENV_PATH}/bin/python"
frozen_output = subprocess.check_output(
    ["uv", "pip", "freeze", "--python", venv_python, "--color=never"],
    text=True,
)
frozen_packages = [line.strip() for line in frozen_output.splitlines()]
print()
print(f"Captured {len(frozen_packages)} package versions")
print(frozen_packages)

# Download wheels for the selected package(s)
local_wheels_dir = Path("/tmp/wheels")
subprocess.run(
    ["pip3", "download", "--no-deps", f"--dest={str(local_wheels_dir)}"] + frozen_packages
)
wheels = list(local_wheels_dir.glob("*.whl"))


# Register plugins and scale cluster

print()
print("Register plugins and scale cluster")
start = time.time()

# Target directory on worker
# Writable dirs include /tmp, /wk, /home/odc
TARGET_DIR = "/tmp/site-packages"

# Pip options here follow those of easi-venv-setup.py
pip_options = [
    "--no-build-isolation",
    "--no-deps",
    f"--target={TARGET_DIR}",
]

# client_e.register_plugin(ForwardOutput())  # optional, for debugging
# Upload wheels to worker's local_directory
for whl in wheels:
    client_e.register_plugin(UploadFile(whl, load=False), name=f"upload_file-{whl.name}")

client_e.register_plugin(PipInstallWheels(pip_options))
client_e.register_plugin(SysPathPlugin(TARGET_DIR))
cluster_e.scale(4)
client_e.wait_for_workers(4)

elapsed = time.time() - start
print(f"[Gateway cluster] Scale and venv setup time: {elapsed:.1f}s for 4 workers")


# Confirm

future = client_e.submit(test_localdirectory)
print()
print("[Gateway cluster] Worker local directory contents:")
for p in future.result():
    print(p)

test_pkg = PACKAGES.split()[0].replace("-", "_")
future = client_e.submit(test_import, test_pkg)
print()
print("[Gateway cluster] Worker import test:")
print(future.result())

future = client_e.submit(test_syspath)
print()
print("[Local cluster] Worker sys.path test:")
for p in future.result():
    print(p)

client_e.close()
cluster_e.close()

## Capturing logs from dask workers

Dask provides a few different methods to capture logs from workers. We trialed them in creating this notebook and note our experience here for reference.

Note that each message from a worker is directed through the scheduler to the client. Many messages from many workers could overload the scheduler so we recommend you use these capabilities as sparingly as feasible. For example, it could be more efficient to query a single worker for its state (e.g., `client.submit(query_fn)`) rather than have all workers reporting much the same information to the scheduler and client.

1. Default scheduler/worker logs
   - `client.get_worker_logs()`
   - captures the worker startup and transitions managed by the scheduler
1. dask.distributed print and warn functions ([ref](https://distributed.dask.org/en/latest/api.html#distributed.print))
   - will log a 'print' or 'warn' event
   - use as a drop-in replacement for standard `print` and `warn` functions
   - functions running on workers would need to import the dask.distributed versions
1. `ForwardOutput` worker plugin ([ref](https://github.com/dask/distributed/blob/main/distributed/diagnostics/plugin.py))
   - forward stdout and stderr from the worker to the client
   - only need to register an additional plugin
1. `client.forward_logging()` ([ref](https://distributed.dask.org/en/latest/api.html#distributed.Client.forward_logging))
   - connect worker-side logger with a client-side logger of the same name
   - client side handler/formatter customisation using the standard python logging module
1. Log events from worker ([ref](https://distributed.dask.org/en/stable/logging.html#structured-logs_]))
   - named events are logged from the worker to the scheduler
   - client can retrieve event logs when desired but workers still log all events to the scheduler

In [ ]:
# Kill your gateways cluster if things get out of hand

# from dask_gateway import Gateway
# gateway = Gateway()
# clusters = gateway.list_clusters()
# print (f'trying to stop {len(clusters)} clusters')
# for c in clusters:
#     gateway.stop_cluster(c.name)